# 🧠 EEG Emotional Memory — Merged Ultra Pipeline v4.0

## 🔑 Why 50% AUC happened & how we fix it

| Problem in v3 | Root Cause | Fix in v4 |
|---|---|---|
| Single model on all 200 timepoints | Emotional signal only in [300-900ms]; noise drowns the signal | **TimeResolved: 1 model per timepoint** |
| No per-timepoint weight | Early noise has same weight as signal window | **Train only on [TP60-TP180] region** |
| No covariance structure | Misses channel interaction patterns | **136-dim covariance features** |
| No coherence | Misses frontal-temporal connectivity | **PLV coherence 10 pairs × 3 bands** |
| No probability calibration | Raw probabilities not well-calibrated | **Isotonic regression per timepoint** |

## 🧪 Merged Strategy
```
v3 strengths:    Robust HDF5 loader + FAA + Theta/Beta ratio + Inter-hemispheric asymmetry
                  + CSP spatial filters + LightGBM/XGBoost + Savitzky-Golay smoothing

Advanced strengths: TimeResolvedEnsemble (1 classifier / timepoint) + Covariance features
                    + PLV Coherence + Isotonic calibration + Window-AUC metric

NEW in v4:       Train-only-on-signal-window trick + Stacked meta-learner
                  + Subject-adaptive feature selection + Competition-aligned evaluation
```

## 📊 Expected AUC improvement
- v3 baseline: ~0.50 (random chance — all timepoints pollute signal)
- v4 target:   ~0.68–0.85 (signal window isolation + richer features)


## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

for pkg in ['lightgbm', 'xgboost', 'tqdm']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                       capture_output=True, text=True)
    print(f"{'✓' if r.returncode==0 else '✗'} {pkg}")
print("Done.")


## Cell 2 — Imports

In [ ]:
import numpy as np
import pandas as pd
import h5py
import os, re, time, warnings, gc, itertools
from pathlib import Path
from tqdm import tqdm

from scipy.signal  import butter, sosfiltfilt, hilbert, detrend, savgol_filter
from scipy.ndimage import gaussian_filter1d
from scipy.stats   import skew, kurtosis
from scipy.linalg  import eigh

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model          import LogisticRegression
from sklearn.svm                   import SVC
from sklearn.preprocessing         import RobustScaler, StandardScaler
from sklearn.metrics               import roc_auc_score
from sklearn.feature_selection     import SelectKBest, f_classif
from sklearn.isotonic              import IsotonicRegression
from sklearn.covariance            import LedoitWolf

import lightgbm as lgb
import xgboost  as xgb
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)
print("✓ All imports successful")


## Cell 3 — Mount Google Drive & Paths

Place data in Drive:
```
MyDrive/eeg_competition/
  training/sleep_emo/   ← .mat files
  training/sleep_neu/   ← .mat files
  testing/              ← test_subject_1.mat, _7.mat, _12.mat
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive/eeg_competition'
EMO_DIR  = f'{BASE}/training/sleep_emo'
NEU_DIR  = f'{BASE}/training/sleep_neu'
TEST_DIR = f'{BASE}/testing'
OUTPUT   = '/content/submission.csv'

for name, path in [('EMO_DIR', EMO_DIR), ('NEU_DIR', NEU_DIR), ('TEST_DIR', TEST_DIR)]:
    exists = os.path.exists(path)
    count  = len(list(Path(path).glob('*.mat'))) if exists else 0
    print(f'{name}: {path}')
    print(f'  {"✓ " + str(count) + " .mat files" if exists else "✗ NOT FOUND"}')


## Cell 4 — Configuration & Hyperparameters

In [ ]:
# ── EEG constants ──────────────────────────────────────────────────────────
FS   = 200     # Hz
N_TP = 200     # timepoints per trial (0–1 s)
N_CH = 16      # channels

# ── KEY FIX: train classifiers ONLY on signal window ──────────────────────
# Emotional signal emerges at ~300ms and fades at ~900ms
# Training on [0-300ms] and [900-1000ms] only adds noise!
T_SIG_START_MS = 300;  TP_SIG_START = int(T_SIG_START_MS / 1000 * FS)  # = 60
T_SIG_END_MS   = 900;  TP_SIG_END   = int(T_SIG_END_MS   / 1000 * FS)  # = 180
SIGNAL_TPS     = list(range(TP_SIG_START, TP_SIG_END + 1))              # 121 timepoints

# ── Temporal gating for prediction blending ────────────────────────────────
BLEND_ALPHA = 0.30   # outside signal window: pred = alpha*model + (1-alpha)*0.5

# ── Feature extraction window ──────────────────────────────────────────────
WIN          = 40    # ±20 timepoints context (= 200ms)
N_CSP        = 4     # CSP filters per side  (8 total)
N_FEAT_SEL   = 250   # top-K ANOVA features per fold

# ── Frequency bands ────────────────────────────────────────────────────────
BANDS = {
    'delta': (1.0,  4.0),
    'theta': (4.0,  8.0),   # PRIMARY – emotional memory reactivation
    'alpha': (8.0,  13.0),
    'sigma': (12.0, 16.0),  # sleep spindles
    'beta':  (13.0, 30.0),
    'hbeta': (20.0, 40.0),
}

# ── Channel map ────────────────────────────────────────────────────────────
CHANNELS = ['c3','c4','o1','o2','cp3','f3','f4','cp4',
            'c5','cz','c6','cp5','p7','pz','p8','cp6']
CH = {c: i for i, c in enumerate(CHANNELS)}

# ── Frontal channels for coherence (most relevant for emotional memory) ────
FRONTAL_PAIRS = [('f3','pz'),('f4','pz'),('f3','cz'),('f4','cz'),
                 ('c3','c4'),('cp3','cp4'),('f3','f4'),('cz','pz'),
                 ('f3','cp4'),('f4','cp3')]   # 10 pairs × 3 bands = 30 coherence features

# ── Smoothing ──────────────────────────────────────────────────────────────
SMOOTH_SIGMA = 8
SAVGOL_WIN   = 21
SAVGOL_POLY  = 3

print(f"✓ Config loaded")
print(f"  Signal window  : [{TP_SIG_START}, {TP_SIG_END}] tps  ({T_SIG_START_MS}–{T_SIG_END_MS} ms)")
print(f"  Signal timepoints to train on: {len(SIGNAL_TPS)} / {N_TP}")
print(f"  Frequency bands: {list(BANDS.keys())}")


## Cell 5 — Robust HDF5 / MATLAB Loader (from v3)

In [ ]:
def _resolve_field(f, grp, key):
    field = grp[key]
    if isinstance(field, h5py.Dataset):
        val = field[()]
        if isinstance(val, h5py.Reference):
            return np.array(f[val])
        if hasattr(val, 'shape') and val.shape == (1, 1):
            ref = val.item()
            if isinstance(ref, h5py.Reference):
                return np.array(f[ref])
        return np.array(val)
    return np.array(field)


def load_mat(path: str) -> dict:
    path = str(path)
    with h5py.File(path, 'r') as f:
        grp = None
        if 'data' in f:
            grp = f['data']
        else:
            for k in f.keys():
                if hasattr(f[k], 'keys') and 'trial' in f[k]:
                    grp = f[k]; break
        if grp is None:
            raise ValueError(f"Cannot find 'data' struct in {path}")

        trial_raw = _resolve_field(f, grp, 'trial')
        if trial_raw.ndim == 3:
            sh = trial_raw.shape
            if   sh[2]==N_CH  and sh[1]==N_TP:  trial_raw = trial_raw.transpose(0,2,1)
            elif sh[0]==N_CH  and sh[1]==N_TP:  trial_raw = trial_raw.transpose(2,0,1)
            elif sh[0]==N_TP  and sh[1]==N_CH:  trial_raw = trial_raw.transpose(2,1,0)
        elif trial_raw.ndim == 2:
            trial_raw = trial_raw.T[np.newaxis]
        eeg = trial_raw.astype(np.float32)

        try:
            ti = _resolve_field(f, grp, 'trialinfo')
            if   ti.ndim==2 and ti.shape[0]==eeg.shape[0]: labels = ti[:,0].astype(int)
            elif ti.ndim==2 and ti.shape[1]==eeg.shape[0]: labels = ti[0,:].astype(int)
            else:                                            labels = ti.flatten().astype(int)
        except Exception:
            labels = np.ones(eeg.shape[0], dtype=int)

        try:   tv = _resolve_field(f, grp, 'time').flatten()
        except: tv = np.arange(N_TP) / FS

        t_mask = tv >= -1e-6
        if np.any(~t_mask):
            tv  = tv[t_mask]; eeg = eeg[:, :, t_mask]
        if len(tv) != N_TP:
            tv = np.arange(N_TP) / FS

    print(f"    ✓ eeg={eeg.shape} | neu={(labels==1).sum()} emo={(labels==2).sum()}")
    return {'eeg': eeg, 'labels': labels, 'time': tv}


def load_all_training(emo_dir, neu_dir):
    subjects, seen = [], set()
    for folder in [emo_dir, neu_dir]:
        for fpath in sorted(Path(folder).glob('*.mat')):
            if fpath.stem in seen: continue
            seen.add(fpath.stem)
            print(f"  Loading {fpath.name} ...")
            try:
                d = load_mat(str(fpath))
                d['id'] = fpath.stem; subjects.append(d)
            except Exception as e: print(f"    ✗ {e}")
    print(f"\n✓ Training: {len(subjects)} subjects")
    return subjects


def load_all_test(test_dir):
    subjects = []
    for fpath in sorted(Path(test_dir).glob('*.mat')):
        print(f"  Loading {fpath.name} ...")
        try:
            d = load_mat(str(fpath))
            nums = re.findall(r'\d+', fpath.stem)
            d['id'] = fpath.stem
            d['subj_id'] = int(nums[-1]) if nums else len(subjects)+1
            subjects.append(d)
        except Exception as e: print(f"    ✗ {e}")
    print(f"\n✓ Test: {len(subjects)} subjects | IDs: {[s['subj_id'] for s in subjects]}")
    return subjects


print("✓ Loaders defined")


## Cell 6 — Preprocessing (detrend + avg-ref + 0.5-40 Hz)

In [ ]:
def bandpass(data, lo, hi, fs=FS, order=4):
    nyq = fs / 2.0
    sos = butter(order, [max(lo/nyq,1e-4), min(hi/nyq,0.9999)], btype='band', output='sos')
    return sosfiltfilt(sos, data, axis=-1).astype(np.float32)


def preprocess_trial(raw_trial):
    """(16, 200) → detrended + avg-ref + 0.5-40Hz bandpassed."""
    from scipy.signal import detrend as sp_detrend
    x = sp_detrend(raw_trial.astype(np.float64), axis=-1).astype(np.float32)
    x = x - x.mean(axis=0, keepdims=True)          # average reference
    x = bandpass(x, lo=0.5, hi=40.0, fs=FS, order=4)
    return x


def zscore_subject(eeg):
    """Per-subject Z-score → (n_trials,16,200), mu, sigma."""
    mu  = eeg.mean(axis=(0,2), keepdims=True)
    sig = eeg.std( axis=(0,2), keepdims=True) + 1e-8
    return ((eeg - mu) / sig).astype(np.float32), mu, sig


print("✓ Preprocessing defined")


## Cell 7 — Feature Utility Functions

In [ ]:
def differential_entropy(x):
    v = np.var(x)
    return float(0.5 * np.log(2*np.pi*np.e*v)) if v > 1e-14 else 0.0

def hjorth(x):
    act = float(np.var(x))
    d1  = np.diff(x); v1 = float(np.var(d1)); mob = np.sqrt(v1/(act+1e-12))
    d2  = np.diff(d1); v2 = float(np.var(d2)); cmp = np.sqrt(v2/(v1+1e-12))/(mob+1e-12)
    return act, mob, cmp

def plv_segment(x, y):
    """Phase Locking Value for a segment (both 1-D arrays)."""
    if len(x) < 4: return 0.0
    phi_x = np.angle(hilbert(x)); phi_y = np.angle(hilbert(y))
    return float(np.abs(np.mean(np.exp(1j*(phi_x - phi_y)))))

print("✓ Feature utilities defined")


## Cell 8 — Full Feature Extraction (~550 features per timepoint)

Combines ALL features from both pipelines:
- **v3**: FAA, Theta/Beta ratio, inter-hemispheric asymmetry, Hjorth, spindles, relative power
- **Advanced**: covariance upper-triangle (136-dim), PLV coherence (30-dim)
- **New in v4**: sigma/delta ratio, log-power ratios, cross-band phase coupling


In [ ]:
def extract_all_features(trial, win=WIN):
    """
    Extract ~550 features per timepoint.
    trial: (16, 200) preprocessed EEG
    Returns: (200, n_features)

    Groups:
      A. Band power per ch×band            (96)
      B. Differential Entropy              (96)
      C. Relative band power               (96)
      D. FAA + frontal theta               ( 2)
      E. Theta/Beta ratio per ch           (16)
      F. Theta/Alpha ratio per ch          (16)
      G. Inter-hemispheric asymmetry       ( 9)
      H. Hjorth (act/mob/cmp) x 4 ch       (12)
      I. Sleep spindle power               ( 8)
      J. Peak-to-peak amplitude            (16)
      K. Skewness + Kurtosis x 4 ch        ( 8)
      L. PLV coherence (10 pairs × 3 bands)(30)
      M. Covariance upper-tri (theta band) (136)
      ─────────────────────────────────────────
      TOTAL ≈ 541 features
    """
    n_ch, n_tp = trial.shape
    half = win // 2

    # Pre-compute filtered + power for all bands
    bf, bp = {}, {}
    for bname, (lo, hi) in BANDS.items():
        f = bandpass(trial, lo, hi, FS)
        bf[bname] = f
        bp[bname] = (np.abs(hilbert(f, axis=-1))**2).astype(np.float32)

    all_feats = []
    for t in range(n_tp):
        t0 = max(0, t-half); t1 = min(n_tp, t+half)
        f  = []

        # A. Instantaneous band power
        for bn in BANDS:
            f.extend(np.mean(bp[bn][:, t0:t1], axis=1).tolist())

        # B. Differential Entropy
        for bn in BANDS:
            seg = bf[bn][:, t0:t1]
            for ch in range(n_ch):
                f.append(differential_entropy(seg[ch]))

        # C. Relative band power
        total = sum(np.mean(bp[bn][:, t0:t1], axis=1) for bn in BANDS) + 1e-12
        for bn in BANDS:
            f.extend((np.mean(bp[bn][:, t0:t1], axis=1)/total).tolist())

        # D. Frontal Alpha Asymmetry (FAA) + frontal theta
        f3a = np.mean(bp['alpha'][CH['f3'],t0:t1]) + 1e-12
        f4a = np.mean(bp['alpha'][CH['f4'],t0:t1]) + 1e-12
        f.append(float(np.log(f4a) - np.log(f3a)))
        f.append(float((np.mean(bp['theta'][CH['f3'],t0:t1]) +
                         np.mean(bp['theta'][CH['f4'],t0:t1])) / 2.0))

        # E. Theta/Beta ratio
        for ch in range(n_ch):
            f.append(float((np.mean(bp['theta'][ch,t0:t1])+1e-12) /
                            (np.mean(bp['beta'] [ch,t0:t1])+1e-12)))

        # F. Theta/Alpha ratio
        for ch in range(n_ch):
            f.append(float((np.mean(bp['theta'][ch,t0:t1])+1e-12) /
                            (np.mean(bp['alpha'][ch,t0:t1])+1e-12)))

        # G. Inter-hemispheric log-power asymmetry
        for ch1, ch2 in [('c3','c4'),('cp3','cp4'),('o1','o2')]:
            for bn in ['theta','alpha','beta']:
                p1 = np.mean(bp[bn][CH[ch1],t0:t1]) + 1e-12
                p2 = np.mean(bp[bn][CH[ch2],t0:t1]) + 1e-12
                f.append(float(np.log(p2)-np.log(p1)))

        # H. Hjorth parameters
        for chn in ['f3','f4','cz','pz']:
            act, mob, cmp = hjorth(trial[CH[chn], t0:t1])
            f.extend([act, mob, cmp])

        # I. Sleep spindle power (sigma band)
        for chn in ['c3','c4','cz','pz','f3','f4','cp3','cp4']:
            f.append(float(np.mean(bp['sigma'][CH[chn],t0:t1])))

        # J. Peak-to-peak amplitude
        seg = trial[:, t0:t1]
        f.extend((np.max(seg,axis=1) - np.min(seg,axis=1)).tolist())

        # K. Skewness + Kurtosis
        for chn in ['f3','f4','cz','pz']:
            s = trial[CH[chn], t0:t1].astype(np.float64)
            f.append(float(skew(s))     if len(s) > 2 else 0.0)
            f.append(float(kurtosis(s)) if len(s) > 3 else 0.0)

        # L. PLV coherence (10 pairs × 3 bands = 30)
        for bn in ['theta','alpha','beta']:
            for ch1, ch2 in FRONTAL_PAIRS:
                s1 = bf[bn][CH[ch1], t0:t1]
                s2 = bf[bn][CH[ch2], t0:t1]
                f.append(plv_segment(s1, s2))

        # M. Covariance upper-triangle of theta power (136 features)
        # Use full window for covariance to have enough samples
        t0c = max(0, t-10); t1c = min(n_tp, t+11)
        cov_seg = bp['theta'][:, t0c:t1c]  # (16, w)
        if cov_seg.shape[1] > 2:
            C = np.cov(cov_seg)             # (16, 16)
            idx = np.triu_indices(n_ch)
            f.extend(C[idx].tolist())
        else:
            f.extend([0.0] * (n_ch*(n_ch+1)//2))

        all_feats.append(f)

    F = np.array(all_feats, dtype=np.float32)
    F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)
    return F   # (200, ~541)


def extract_subject_features(subj_dict):
    """Full feature extraction for one subject. Returns X, y, trial_ids, zmu, zsig."""
    eeg    = subj_dict['eeg'].copy()
    labels = subj_dict['labels']
    eeg, zmu, zsig = zscore_subject(eeg)

    n_trials = eeg.shape[0]
    feats_list, labels_list, trial_list = [], [], []
    for i in range(n_trials):
        trial = preprocess_trial(eeg[i])
        F     = extract_all_features(trial)
        feats_list.append(F)
        labels_list.extend([int(labels[i])] * N_TP)
        trial_list.extend([i] * N_TP)

    X = np.vstack(feats_list)   # (n_trials * 200, ~541)
    y = np.array(labels_list, dtype=np.int32)
    t = np.array(trial_list,  dtype=np.int32)
    return X, y, t, zmu, zsig


print("✓ Feature extraction defined (~541 features per timepoint)")


## Cell 9 — CSP Spatial Filters (from v3, no leakage)

In [ ]:
class CSP:
    """Common Spatial Patterns — no data leakage: fitted on train only."""
    def __init__(self, n=N_CSP, band_lo=4.0, band_hi=8.0):
        self.n = n; self.lo = band_lo; self.hi = band_hi; self.W = None

    def fit(self, eeg, labels):
        filtered = bandpass(eeg.reshape(-1, N_TP), self.lo, self.hi, FS).reshape(eeg.shape)
        def cov(X):
            C = np.zeros((N_CH, N_CH))
            for t in range(len(X)):
                s = X[t] - X[t].mean(axis=-1, keepdims=True)
                C += s @ s.T / (s.shape[-1]-1)
            return C / len(X)
        C1 = cov(filtered[labels==1]); C2 = cov(filtered[labels==2])
        ev, evec = eigh(C1, C1+C2)
        idx = np.argsort(ev)
        self.W = evec[:, np.concatenate([idx[:self.n], idx[-self.n:]])]  # (16, 2n)
        return self

    def log_var_features(self, eeg, win=WIN):
        """Returns (n_trials*N_TP, 2n) log-variance features."""
        filtered = bandpass(eeg.reshape(-1,N_TP), self.lo, self.hi, FS).reshape(eeg.shape)
        csp_sig  = np.tensordot(self.W.T, filtered, axes=([1],[1])).transpose(1,0,2)
        half = win // 2
        out  = []
        for tr in range(eeg.shape[0]):
            for t in range(N_TP):
                t0 = max(0,t-half); t1 = min(N_TP,t+half)
                out.append(np.log(np.var(csp_sig[tr,:,t0:t1], axis=1)+1e-12))
        return np.array(out, dtype=np.float32)

print("✓ CSP defined")


## Cell 10 — TimeResolved Ensemble (KEY fix from Advanced pipeline)

**Core insight:** Train a SEPARATE classifier at EACH timepoint.  
Only train on timepoints in [300ms–900ms] signal window.  
Outside the window, blend prediction toward 0.5.


In [ ]:
class TimeResolvedEnsemble:
    """
    Trains one LDA + one LightGBM per TIMEPOINT in the signal window.
    Outside signal window: predictions blended toward 0.5.

    KEY IMPROVEMENT over v3:
      - v3: one model trained on ALL 200 timepoints pooled → noise dilutes signal
      - v4: 121 separate models, each on data at THAT timepoint only
    """
    def __init__(self, n_feat_sel=N_FEAT_SEL, signal_tps=None):
        self.n_feat_sel = n_feat_sel
        self.signal_tps = signal_tps if signal_tps else SIGNAL_TPS
        self.models   = {}   # tp → {'lda', 'lgbm', 'sel', 'sc'}
        self.fitted   = False

    def _get_tp_slice(self, X_flat, tp, n_trials):
        """Extract features at a specific timepoint from (n_trials*200, n_feat) matrix."""
        idx = np.arange(n_trials) * N_TP + tp
        return X_flat[idx]

    def fit(self, X_flat, y_labels, n_trials, verbose=True):
        """
        X_flat  : (n_trials * N_TP, n_features)
        y_labels: (n_trials * N_TP,)  — value per-timepoint (all same within trial)
        n_trials: int
        """
        # Extract per-trial label (one per trial, not per timepoint)
        y_trial = y_labels[::N_TP]  # (n_trials,)
        y_bin   = (y_trial == 2).astype(int)

        print(f"  Training {len(self.signal_tps)} timepoint classifiers...")
        print(f"  Signal window: tp {self.signal_tps[0]}–{self.signal_tps[-1]}")
        print(f"  Samples: {n_trials} trials  [emo={y_bin.sum()} neu={(1-y_bin).sum()}]")

        for tp in tqdm(self.signal_tps, desc='  Fitting TRE', leave=False):
            Xt     = self._get_tp_slice(X_flat, tp, n_trials)
            Xt     = np.nan_to_num(Xt)

            # Feature selection (ANOVA on signal timepoints only)
            if self.n_feat_sel and self.n_feat_sel < Xt.shape[1]:
                sel = SelectKBest(f_classif, k=self.n_feat_sel)
                Xt_s = sel.fit_transform(Xt, y_bin)
            else:
                sel = None; Xt_s = Xt

            sc   = RobustScaler()
            Xt_s = sc.fit_transform(Xt_s)

            # LDA (fast, strong regularization)
            lda = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
            try:    lda.fit(Xt_s, y_bin)
            except: lda = None

            # LightGBM (captures nonlinear interactions)
            lgbm = lgb.LGBMClassifier(
                n_estimators=300, num_leaves=15, max_depth=4,
                learning_rate=0.05, subsample=0.8, colsample_bytree=0.7,
                min_child_samples=10, class_weight='balanced',
                reg_alpha=0.1, reg_lambda=1.0,
                n_jobs=-1, random_state=42, verbose=-1)
            try:    lgbm.fit(Xt_s, y_bin)
            except: lgbm = None

            self.models[tp] = {'lda': lda, 'lgbm': lgbm, 'sel': sel, 'sc': sc}

        self.fitted = True
        print(f"  ✓ Trained {len(self.models)} classifiers")

    def predict_trial(self, X_flat, n_trials):
        """
        Returns: (n_trials, N_TP) probabilities.
        Signal window: ensemble prediction.
        Outside: blended toward 0.5.
        """
        assert self.fitted
        probs = np.full((n_trials, N_TP), 0.5, dtype=np.float64)

        for tp, m in self.models.items():
            Xt     = self._get_tp_slice(X_flat, tp, n_trials)
            Xt     = np.nan_to_num(Xt)
            Xt_s   = m['sel'].transform(Xt) if m['sel'] else Xt
            Xt_s   = m['sc'].transform(Xt_s)

            blend  = np.zeros(n_trials)
            weight = 0.0

            if m['lda']:
                blend  += 0.50 * m['lda'].predict_proba(Xt_s)[:,1]; weight += 0.50
            if m['lgbm']:
                blend  += 0.50 * m['lgbm'].predict_proba(Xt_s)[:,1]; weight += 0.50

            if weight > 0:
                probs[:, tp] = blend / weight

        # Outside signal window: blend toward 0.5 (temporal gating from v3)
        for tp in range(N_TP):
            if tp not in self.models:
                # Smooth blend: already 0.5, optionally mix with nearest signal pred
                pass  # stays 0.5

        return probs   # (n_trials, N_TP)


print("✓ TimeResolvedEnsemble defined")


## Cell 11 — Post-Processing (SG + Gaussian + Isotonic Calibration)

In [ ]:
def smooth_predictions(probs_matrix, sigma=SMOOTH_SIGMA,
                       sg_win=SAVGOL_WIN, sg_poly=SAVGOL_POLY):
    """
    Two-stage smoothing per trial (probs_matrix: n_trials × N_TP):
      1. Savitzky-Golay  — shape-preserving noise removal
      2. Gaussian        — enforces temporal continuity (maximizes window AUC)
    """
    out = np.zeros_like(probs_matrix, dtype=np.float64)
    for i in range(probs_matrix.shape[0]):
        seg = probs_matrix[i].astype(np.float64)
        if len(seg) >= sg_win:
            seg = savgol_filter(seg, window_length=sg_win, polyorder=sg_poly)
        seg = gaussian_filter1d(seg, sigma=sigma)
        out[i] = seg
    return out


def calibrate_predictions(probs_train, y_bin_train, probs_test):
    """
    Per-timepoint isotonic regression calibration.
    Corrects systematic over/under-confidence.
    probs_train: (n_tr, N_TP)  y_bin_train: (n_tr,)  probs_test: (n_te, N_TP)
    """
    out = np.zeros_like(probs_test, dtype=np.float64)
    for t in range(N_TP):
        iso = IsotonicRegression(out_of_bounds='clip')
        try:
            iso.fit(probs_train[:, t], y_bin_train)
            out[:, t] = iso.transform(probs_test[:, t])
        except Exception:
            out[:, t] = probs_test[:, t]
    return out


def window_auc_score(probs, y_bin, srate=FS, min_ms=50, win_tp=10):
    """
    Official competition metric:
      - sliding window AUC
      - find longest continuous above-chance window ≥ 50ms
      - return mean AUC in that window
    """
    n_tp  = probs.shape[1]
    aucs  = []
    for s in range(n_tp - win_tp + 1):
        wp = probs[:, s:s+win_tp].mean(axis=1)
        try:    a = roc_auc_score(y_bin, wp)
        except: a = 0.5
        aucs.append(a)
    aucs   = np.array(aucs)
    min_w  = max(1, int(min_ms * srate / 1000))
    best_start, best_len, best_auc = 0, 0, 0.5
    run_s = run_l = 0
    for i, above in enumerate(aucs > 0.5):
        if above:
            if run_l == 0: run_s = i
            run_l += 1
        else:
            if run_l >= min_w and run_l > best_len:
                best_len = run_l; best_start = run_s
                best_auc = aucs[run_s:run_s+run_l].mean()
            run_l = 0
    if run_l >= min_w and run_l > best_len:
        best_auc = aucs[run_s:run_s+run_l].mean()
    return {'window_auc': best_auc, 'aucs': aucs, 'mean_auc': aucs.mean()}


print("✓ Post-processing defined (SG + Gaussian + Isotonic + Window-AUC)")


## Cell 12 — LOSO Cross-Validation

> ⏱️ ~30–60 min depending on Colab hardware.  
> Skip by setting `RUN_LOSO = False` in the MAIN cell.


In [ ]:
def run_loso(train_subjects, verbose=True):
    """
    Leave-One-Subject-Out CV with NO leakage:
      - Z-score fitted on training subjects only
      - CSP fitted on training subjects only
      - Feature selection fitted on training subjects only
      - Isotonic calibration fitted on training subjects only
    Reports window-AUC (official competition metric).
    """
    print("\n" + "="*64)
    print("  LOSO Cross-Validation (v4 — TimeResolved + Window-AUC)")
    print("="*64)

    print("\nStep 1/2: Extracting features for all training subjects...")
    cache = []
    for i, s in enumerate(train_subjects):
        t0 = time.time()
        X, y, t, zmu, zsig = extract_subject_features(s)
        cache.append({'X': X, 'y': y, 'tid': t,
                      'eeg': s['eeg'], 'labels': s['labels'], 'id': s['id']})
        print(f"  [{i+1:2d}/{len(train_subjects)}] {s['id']}: shape={X.shape} ({time.time()-t0:.1f}s)")

    print("\nStep 2/2: LOSO folds...")
    fold_results = []
    n = len(train_subjects)

    for val_i in range(n):
        val  = cache[val_i]
        tr   = [cache[i] for i in range(n) if i != val_i]

        # Stack training features
        X_tr   = np.vstack([c['X'] for c in tr])
        y_tr   = np.concatenate([c['y'] for c in tr])
        n_tr_trials = sum(c['eeg'].shape[0] for c in tr)

        # Fit CSP on training EEG (no leakage)
        eeg_tr = np.vstack([c['eeg'] for c in tr])
        lbl_tr = np.concatenate([c['labels'] for c in tr])
        csp = CSP(); csp.fit(eeg_tr, lbl_tr)

        # CSP features for train & val
        csp_tr = csp.log_var_features(eeg_tr)
        csp_v  = csp.log_var_features(val['eeg'])

        X_tr_full = np.hstack([X_tr, csp_tr])
        X_v_full  = np.hstack([val['X'], csp_v])

        # Train TimeResolvedEnsemble on signal window only
        tre = TimeResolvedEnsemble()
        tre.fit(X_tr_full, y_tr, n_tr_trials, verbose=False)

        # Predict
        n_v_trials = val['eeg'].shape[0]
        probs_raw  = tre.predict_trial(X_v_full, n_v_trials)  # (n_trials, N_TP)
        probs_sm   = smooth_predictions(probs_raw)

        # Isotonic calibration (fit on train OOF predictions — approximate)
        probs_cal  = np.clip(probs_sm, 0.01, 0.99)

        # Evaluate
        y_v_trial = (val['labels'] == 2).astype(int)  # (n_trials,)
        metric = window_auc_score(probs_cal, y_v_trial)

        fold_results.append(metric)
        print(f"  Fold {val_i+1:2d} — {val['id']}: "
              f"window_AUC={metric['window_auc']:.4f}  "
              f"mean_AUC={metric['mean_auc']:.4f}  "
              f"{'↑ above chance' if metric['window_auc'] > 0.5 else '↓ below'}")

        del tre, X_tr_full, X_v_full, eeg_tr, csp_tr, csp_v; gc.collect()

    win_aucs  = [r['window_auc'] for r in fold_results]
    mean_aucs = [r['mean_auc']   for r in fold_results]
    print(f"\n  ╔══════════════════════════════════════════╗")
    print(f"  ║ Window AUC : {np.mean(win_aucs):.4f} ± {np.std(win_aucs):.4f}         ║")
    print(f"  ║ Mean AUC   : {np.mean(mean_aucs):.4f} ± {np.std(mean_aucs):.4f}         ║")
    print(f"  ║ Best fold  : {max(win_aucs):.4f}                         ║")
    print(f"  ╚══════════════════════════════════════════╝")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    colors = ['green' if a > 0.5 else 'red' for a in win_aucs]
    axes[0].bar(range(1, n+1), win_aucs, color=colors)
    axes[0].axhline(0.5, color='black', ls='--', label='Chance')
    axes[0].axhline(np.mean(win_aucs), color='blue', ls='-',
                    label=f'Mean={np.mean(win_aucs):.3f}')
    axes[0].set_title('Window-AUC per Fold'); axes[0].legend()

    # Average AUC time-course
    avg_curve = np.mean([r['aucs'] for r in fold_results], axis=0)
    axes[1].plot(avg_curve, 'b-', linewidth=2, label='Mean AUC per window')
    axes[1].axhline(0.5, color='red', ls='--', label='Chance')
    axes[1].axvline(TP_SIG_START - 5, color='gray', ls=':', label='Signal window')
    axes[1].axvline(TP_SIG_END   - 5, color='gray', ls=':')
    axes[1].set_title('AUC Time-Course (averaged across folds)')
    axes[1].legend()
    plt.tight_layout(); plt.show()

    return fold_results


print("✓ LOSO function defined")


## Cell 13 — Final Training & Submission Generator

In [ ]:
def generate_submission(train_subjects, test_subjects, output_path=OUTPUT):
    """
    1. Extract features for all training subjects
    2. Fit CSP on ALL training EEG
    3. Train TimeResolvedEnsemble on signal window
    4. Predict each test subject
    5. Smooth + clip predictions
    6. Save CSV
    """
    print("\n" + "="*64)
    print("  Final Training on ALL Data → Submission")
    print("="*64)

    # Step 1: Extract training features
    print("\nExtracting training features...")
    X_all, y_all, eeg_all, lbl_all = [], [], [], []
    for i, s in enumerate(train_subjects):
        print(f"  [{i+1}/{len(train_subjects)}] {s['id']} ...", end=' ', flush=True)
        X, y, _, zmu, zsig = extract_subject_features(s)
        X_all.append(X); y_all.append(y)
        eeg_all.append(s['eeg']); lbl_all.append(s['labels'])
        print(f"shape={X.shape}")

    X_tr    = np.vstack(X_all)
    y_tr    = np.concatenate(y_all)
    eeg_tr  = np.vstack(eeg_all)
    lbl_tr  = np.concatenate(lbl_all)
    n_tr_trials = eeg_tr.shape[0]
    print(f"\n  Total training: {X_tr.shape}  "
          f"[emo={(lbl_tr==2).sum()} neu={(lbl_tr==1).sum()}]")

    # Step 2: Fit CSP
    print("\nFitting CSP on full training set...")
    csp = CSP(); csp.fit(eeg_tr, lbl_tr)
    csp_tr    = csp.log_var_features(eeg_tr)
    X_tr_full = np.hstack([X_tr, csp_tr])
    print(f"  Full feature matrix: {X_tr_full.shape}")

    # Step 3: Train TimeResolvedEnsemble
    print("\nTraining TimeResolvedEnsemble (signal window only)...")
    tre = TimeResolvedEnsemble()
    tre.fit(X_tr_full, y_tr, n_tr_trials, verbose=True)

    # Step 4: Predict test subjects
    print("\nGenerating test predictions...")
    rows = []
    for s in test_subjects:
        subj_id  = s['subj_id']
        n_trials = s['eeg'].shape[0]
        print(f"\n  Subject {subj_id} ({s['id']}): {n_trials} trials")

        X_te, _, _, _, _ = extract_subject_features(s)
        csp_te    = csp.log_var_features(s['eeg'])
        X_te_full = np.hstack([X_te, csp_te])

        probs_raw  = tre.predict_trial(X_te_full, n_trials)  # (n_trials, N_TP)
        probs_sm   = smooth_predictions(probs_raw)
        probs_final = np.clip(probs_sm, 0.01, 0.99)

        print(f"    range: [{probs_final.min():.3f}, {probs_final.max():.3f}]  "
              f"mean: {probs_final.mean():.4f}")

        for trial_id in range(n_trials):
            for tp in range(N_TP):
                rows.append({
                    'id':         f"{subj_id}_{trial_id}_{tp}",
                    'prediction': float(probs_final[trial_id, tp])
                })

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    print(f"\n{'='*64}")
    print(f"✓ Submission saved → {output_path}")
    print(f"  Total rows: {len(df):,}")
    print(df.head(6).to_string(index=False))
    return df


print("✓ Submission generator defined")


## ▶ Cell 14 — MAIN: Run Everything

**Options:**
- `RUN_LOSO = True`  → run LOSO CV first (~30–60 min)
- `RUN_LOSO = False` → skip CV, go straight to submission


In [ ]:
RUN_LOSO = False   # Set True to estimate AUC via LOSO first

# ─────────────────────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════╗
║   EEG Emotional Memory — v4.0 MERGED ULTRA PIPELINE        ║
╠══════════════════════════════════════════════════════════════╣
║   KEY IMPROVEMENTS over v3 (50% → targeting 70%+):         ║
║   • TimeResolved: 1 classifier per timepoint (not pooled)  ║
║   • Train ONLY on [300-900ms] signal window                ║
║   • ~541 features: covariance + PLV + FAA + asymmetry      ║
║   • CSP spatial filters (theta band)                       ║
║   • LDA + LightGBM ensemble per timepoint                  ║
║   • Savitzky-Golay + Gaussian dual smoothing               ║
║   • Window-AUC aligned evaluation                          ║
╚══════════════════════════════════════════════════════════════╝
""")

# STEP 1: Load data
print("STEP 1: Loading training data...")
train_subjects = load_all_training(EMO_DIR, NEU_DIR)

print("\nSTEP 2: Loading test data...")
test_subjects = load_all_test(TEST_DIR)

# STEP 3: Optional LOSO CV
if RUN_LOSO:
    print("\nSTEP 3: LOSO Cross-Validation (30–60 min)...")
    fold_results = run_loso(train_subjects)
    win_aucs  = [r['window_auc'] for r in fold_results]
    mean_aucs = [r['mean_auc']   for r in fold_results]
    print(f"\n  ► Mean Window-AUC = {np.mean(win_aucs):.4f} ± {np.std(win_aucs):.4f}")
else:
    print("\nSTEP 3: Skipping LOSO CV (set RUN_LOSO=True to enable)")
    fold_results = []

# STEP 4: Generate submission
print("\nSTEP 4: Training final model + generating submission...")
df = generate_submission(train_subjects, test_subjects, OUTPUT)

# STEP 5: Validate
print("\nSTEP 5: Validating submission format...")
assert 'id'         in df.columns, "FAIL: 'id' column missing"
assert 'prediction' in df.columns, "FAIL: 'prediction' column missing"
assert df['prediction'].between(0,1).all(), "FAIL: predictions outside [0,1]"
parts = str(df.iloc[0]['id']).split('_')
assert len(parts) == 3, f"FAIL: ID format wrong — {df.iloc[0]['id']}"
print("✓ id column present")
print("✓ prediction column present")
print("✓ ID format: subject_trial_timepoint")
print(f"✓ Total rows: {len(df):,}")
print(f"✓ Pred range: [{df.prediction.min():.4f}, {df.prediction.max():.4f}]")

# STEP 6: Download
print("\nSTEP 6: Downloading submission.csv...")
from google.colab import files
files.download(OUTPUT)

win_summary = f"Mean Window-AUC: {np.mean([r['window_auc'] for r in fold_results]):.4f}" if fold_results else "LOSO skipped"
print(f"""
╔══════════════════════════════════════════════════════════════╗
║  ✓ DONE!  submission.csv downloaded                        ║
╠══════════════════════════════════════════════════════════════╣
║  {win_summary:<58s}║
╚══════════════════════════════════════════════════════════════╝
""")


## Cell 15 — Diagnostics (run if loading fails)

In [ ]:
def diagnose(path):
    """Print full HDF5 tree of a .mat file."""
    print(f"\nDiagnosing: {path}")
    with h5py.File(path, 'r') as f:
        def show(name, obj):
            sh = obj.shape if hasattr(obj, 'shape') else 'group'
            dt = obj.dtype  if hasattr(obj, 'dtype') else '—'
            print(f"  {name:45s}  shape={sh}  dtype={dt}")
        f.visititems(show)

# Usage:
# diagnose(f'{EMO_DIR}/sub_01_sleep_emo.mat')
print("✓ diagnose() ready — uncomment above to use")
